<a href="https://colab.research.google.com/github/julmiha25-sys/MathStatistica/blob/main/%D0%A1%D1%82%D0%B0%D1%82%D0%B8%D1%81%D1%82%D0%B8%D0%BA%D0%B0_%D0%BF%D1%80%D0%BE%D0%B4%D0%B0%D0%B6/%D0%90%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7_%D0%BA%D0%B0%D1%87%D0%B5%D1%81%D1%82%D0%B2%D0%B5%D0%BD%D0%BD%D1%8B%D1%85_%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85/%D0%A1%D1%82%D0%B0%D1%82%D0%B8%D1%81%D1%82%D0%B8%D0%BA%D0%B0_%D0%BF%D1%80%D0%BE%D0%B4%D0%B0%D0%B6_%D0%B0%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7_%D0%BA%D0%B0%D1%87%D0%B5%D1%81%D1%82%D0%B2%D0%B5%D0%BD%D0%BD%D1%8B%D1%85_%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [216]:
# Определение зависимости между сегментом (Segment) и регионом (Region)
import pandas as pd
import numpy as np
from math import *
import scipy.stats
from scipy.stats import chi2_contingency
Orders = pd.read_excel('Sample - Superstore.xlsx', sheet_name='Orders')
Orders = Orders.dropna()
count = Orders.drop_duplicates(subset=['Order ID']).groupby(['Segment', 'Region'])['Segment'].count().unstack()
# Критерий согласия хи-квадрат Пирсона
print(count, chi2_contingency(count) )
# pvalue=0.54 > 0.05 - гипотезу о независимости сегмента и региона не отвергаем
# Проверяем силу связи
N = Orders.drop_duplicates(subset=['Order ID'])['Order ID'].count()
# Кол-во строк (столько различных значений принимает признак "Сегмент")
m = np.shape(count)[0]
# Кол-во столбцов (столько различных значений принимает признак "Регион")
n = np.shape(count)[1]
# Коэффициенты связи Чупрова и Крамера
T = sqrt(chi2_contingency(count).statistic/(N*(m-1)*(n-1)))
K = sqrt(chi2_contingency(count).statistic/(N*min((m-1),(n-1))))
print(T,K)
# T<1 K<1 - связь слабая

Region       Central  East  South  West
Segment                                
Consumer         604   713    444   825
Corporate        348   434    247   485
Home Office      223   254    131   301 Chi2ContingencyResult(statistic=np.float64(4.459073915065623), pvalue=np.float64(0.6148046983206052), dof=6, expected_freq=array([[606.61808744, 723.29526852, 424.37452585, 831.71211819],
       [355.15072869, 423.46057097, 248.45438211, 486.93431823],
       [213.23118387, 254.24416051, 149.17109203, 292.35356359]]))
0.012180670484363599 0.02109754014917236


In [231]:
#2016 году количество продаж по категориям распределяться следующим образом:
#Furniture - 26%, Office Supplies - 43% и Technology - 31%. Проверим, оправдались ли ожидания.
import pandas as pd
a=[0.26,0.43,0.31]
Orders = pd.read_excel('Sample - Superstore.xlsx', sheet_name='Orders')
Orders = Orders.dropna()
Orders['Order Date']=pd.to_datetime(Orders['Order Date']).dt.year
Orders_2016 = Orders[Orders['Order Date'] == 2016]
# Абсолютные наблюдаемые частоты
count = Orders_2016.groupby(['Category'], group_keys=False)['Segment'].count()
N = count.sum()
# Теоретические абсолютные частоты
expected = [a[i] * N for i in range(len(a))]
contingency_table = [count.values, expected]
print("Наблюдаемые абс.частоты | Теоретические абс.частоты")
for obs, exp in zip(count.values, expected):
    print(f"{obs:>23} | {exp:<20.2f}")
print(chi2_contingency(contingency_table))
# pvalue=0 < 0.05 гипотезу об оправдании ожиданий отклоняем

Наблюдаемые абс.частоты | Теоретические абс.частоты
                    561 | 672.36              
                   1566 | 1111.98             
                    459 | 801.66              
Chi2ContingencyResult(statistic=np.float64(180.16685276998606), pvalue=np.float64(7.53815407772066e-40), dof=2, expected_freq=array([[ 616.68, 1338.99,  630.33],
       [ 616.68, 1338.99,  630.33]]))


In [236]:
# Предполагается, что количество заказов по типу доставки будет распределено по регионам одинаково так:
# Standard Class - 60%, Standard Class - 20%, First Class - 15%, Same Day - 5%. Проверим, так ли это на самом деле.
import pandas as pd
from scipy.stats import chi2_contingency
Orders=pd.read_excel('Sample - Superstore.xlsx',sheet_name='Orders')
count = Orders.drop_duplicates(subset=['Order ID']).groupby(['Region','Ship Mode'])['Order ID'].count().reset_index()
count['Наблюдаемые частоты (fact)'] = count.groupby('Region')['Order ID'].transform(lambda x: x / x.sum())
a=[0.6,0.2,0.15,0.05]
count['Ожидаемые частоты (a)'] = count['Ship Mode'].map({'Standard Class':0.6, 'Second Class':0.2, 'First Class':0.15, 'Same Day':0.05})
from scipy.stats import chisquare
# Для каждого региона отдельно
for region in count['Region'].unique():
    obs = count[count['Region']==region]['Order ID'].values
    exp = count[count['Region']==region]['Ожидаемые частоты (a)'].values * obs.sum()
    print(region,':',chisquare(obs, exp))
# pvalue у всех регионов > 0.05 - гипотезу об ожидании распределения во всех регионах доли 60/20/15/5 не отвергаем
table = pd.crosstab(Orders.drop_duplicates(subset=['Order ID'])['Region'],
                    Orders.drop_duplicates(subset=['Order ID'])['Ship Mode'])[['Standard Class', 'Second Class', 'First Class', 'Same Day']]
# Критерий согласия хи-квадрат Пирсона
print(chi2_contingency(table))
# p=0.967 > 0.05 - между регионами нет значимых различий - гипотезу о равенстве распределения по регионам не отвергаем

Central : Power_divergenceResult(statistic=np.float64(0.8652482269503545), pvalue=np.float64(0.8338050615310277))
East : Power_divergenceResult(statistic=np.float64(3.61575065429455), pvalue=np.float64(0.306057026258618))
South : Power_divergenceResult(statistic=np.float64(0.11881589618815924), pvalue=np.float64(0.9894875741460981))
West : Power_divergenceResult(statistic=np.float64(2.3167804676184582), pvalue=np.float64(0.5093138394838377))
Chi2ContingencyResult(statistic=np.float64(2.9246246480923754), pvalue=np.float64(0.967208418032221), dof=9, expected_freq=array([[702.32581354, 226.13296067, 184.61269715,  61.92852865],
       [837.41145937, 269.62747055, 220.12118187,  73.8398882 ],
       [491.32920743, 158.19684568, 129.15032941,  43.32361749],
       [962.93351966, 310.0427231 , 253.11579158,  84.90796566]]))


In [247]:
# Определение зависимости между консультантами и количеством возвратов по сегментам
import pandas as pd
Orders=pd.read_excel('Sample - Superstore.xlsx',sheet_name='Orders')
Returns=pd.read_excel('Sample - Superstore.xlsx',sheet_name='Returns')
People=pd.read_excel('Sample - Superstore.xlsx',sheet_name='People')
df = Orders[Orders['Order ID'].isin(Returns['Order ID'])][['Order ID', 'Region', 'Segment']].drop_duplicates(subset=['Order ID']).reset_index(drop=True)
df=df.merge(People[['Region', 'Person']], on='Region', how='left')
table = pd.crosstab(df['Person'], df['Segment'])
print(table)
# Критерий согласия хи-квадрат Пирсона
print(chi2_contingency(table))
# p_value=0.73 > 0.05 - ависимости между консультантами и сегментами покупателей нет- гипотезу не отвергаем
# Общее количество наблюдений
N = table.values.sum()
# Число строк и столбцов
m = table.shape[0]  # строки (консультанты)
n = table.shape[1]  # столбцы (сегменты)
# Коэффициент Чупрова
T = np.sqrt(chi2 / (N * (m - 1) * (n - 1)))
# Коэффициент Крамера
K = np.sqrt(chi2 / (N * min(m - 1, n - 1)))
print(T,K)
# T=0.066 <1 K=0.11 < 1 - связь слабая - Связь между консультантами и сегментами покупателей практически отсутствует


Segment            Consumer  Corporate  Home Office
Person                                             
Anna Andreadi            96         63           30
Cassandra Brandow        15          6            3
Chuck Magee              25         10            9
Kelly Williams           18         14            7
Chi2ContingencyResult(statistic=np.float64(3.597490170504713), pvalue=np.float64(0.7309571087981459), dof=6, expected_freq=array([[98.33108108, 59.38175676, 31.28716216],
       [12.48648649,  7.54054054,  3.97297297],
       [22.89189189, 13.82432432,  7.28378378],
       [20.29054054, 12.25337838,  6.45608108]]))
0.06611988988640075 0.11452300867410566


In [256]:
# В 2016 году проходила масштабная акция. Определим, был ли эффект по увеличению количества выкупленных товаров.
# Для сравнения возьмем такой же период за предыдущий год.
import pandas as pd
from scipy.stats import chi2_contingency
from scipy.stats import fisher_exact
Orders=pd.read_excel("Sample - Superstore.xlsx",sheet_name='Orders')
Returns=pd.read_excel("Sample - Superstore.xlsx",sheet_name='Returns')
merged = pd.merge(Orders, Returns, on='Order ID', how='left')
grouped = merged.groupby('Order ID')[['Order Date', 'Returned']].max()
grouped['Returned'] = grouped['Returned'].fillna('No')
# Ваши фильтры
Orders_2015 = grouped[(pd.to_datetime(grouped['Order Date']).dt.year == 2015) & (pd.to_datetime(grouped['Order Date']).dt.month <= 11)]['Returned'].value_counts().get('No', 0)
Return_2015 = grouped[(pd.to_datetime(grouped['Order Date']).dt.year == 2015) & (pd.to_datetime(grouped['Order Date']).dt.month <= 11)]['Returned'].value_counts().get('Yes', 0)
Orders_2016 = grouped[(pd.to_datetime(grouped['Order Date']).dt.year == 2016) & (pd.to_datetime(grouped['Order Date']).dt.month <= 11)]['Returned'].value_counts().get('No', 0)
Return_2016 = grouped[(pd.to_datetime(grouped['Order Date']).dt.year == 2016) & (pd.to_datetime(grouped['Order Date']).dt.month <= 11)]['Returned'].value_counts().get('Yes', 0)
# Критерий Фишера из-за малых частот
table = [[Orders_2015, Return_2015], [Orders_2016, Return_2016]]
fisher_exact(table, alternative='less')
# p_value=0.68 > 0.05 - Акция не дала статистически значимого эффекта по увеличению количества выкупленных товаров

SignificanceResult(statistic=np.float64(1.0794270833333333), pvalue=np.float64(0.6866753837812154))

In [261]:
# С ноября 2016 по январь 2017 года проводился тест: выбрали 20 клиентов случайным образом, которые покупали ранее:
# {'Rob Beeghly', 'Aaron Bergman', 'Aaron Smayling', 'Adam Hart', 'Adrian Hane', 'Aimee Bixby', 'Alan Dominguez', 'Alan Haines',
# 'Adrian Shami', 'Katrina Edelman', 'Kelly Collister', 'Kunst Miller', 'Lena Radford', 'Lindsay Shagiari', 'Shaun Chance', 'Sheri Gordon',
# 'Tamara Dahlen', 'Thomas Seio' , 'Tim Brockman', 'Xylona Preis'}.
# В ноябре 2016 им отправили письмо с акцией "Бесплатная доставка", в декабре 2016 - предложили скидку, а в январе 2017 - акцию 3 по цене 2.
# Если хотя бы один заказ был совершен, то это успех, если нет, то неудача.
# Оцените, одинаковый ли эффект дали эти акции? Если разный, то между какими акциями наблюдается различие?
import pandas as pd
Orders=pd.read_excel("Sample - Superstore.xlsx",sheet_name="Orders")
s=['Rob Beeghly', 'Aaron Bergman', 'Aaron Smayling', 'Adam Hart', 'Adrian Hane', 'Aimee Bixby', 'Alan Dominguez', 'Alan Haines',
 'Adrian Shami', 'Katrina Edelman', 'Kelly Collister', 'Kunst Miller', 'Lena Radford', 'Lindsay Shagiari', 'Shaun Chance', 'Sheri Gordon',
 'Tamara Dahlen', 'Thomas Seio' , 'Tim Brockman', 'Xylona Preis']
Orders=Orders[(Orders['Customer Name'].isin(s)) &  (~Orders['Order ID'].isin(Returns['Order ID']))]
Returns=pd.read_excel("Sample - Superstore.xlsx",sheet_name='Returns')
Dostavka = Orders[(pd.to_datetime(Orders['Order Date']).dt.year == 2016) &
                  (pd.to_datetime(Orders['Order Date']).dt.month == 11)].groupby('Customer Name')['Order ID'].count().reset_index(name='Доставка')
Skidka=Orders[(pd.to_datetime(Orders['Order Date']).dt.year == 2016) &
                  (pd.to_datetime(Orders['Order Date']).dt.month == 12)].groupby('Customer Name')['Order ID'].count().reset_index(name='Скидка')
Action=Orders[(pd.to_datetime(Orders['Order Date']).dt.year == 2017) &
                  (pd.to_datetime(Orders['Order Date']).dt.month == 1)].groupby('Customer Name')['Order ID'].count().reset_index(name='Акция')
All = pd.DataFrame(s, columns=['Customer Name'])
All = All.sort_values('Customer Name').reset_index(drop=True)
All = All.merge(Dostavka[['Customer Name', 'Доставка']], on='Customer Name', how='left')
All = All.merge(Skidka[['Customer Name', 'Скидка']], on='Customer Name', how='left')
All = All.merge(Action[['Customer Name', 'Акция']], on='Customer Name', how='left')
All=All.fillna(0)
All[['Доставка', 'Скидка', 'Акция']] = All[['Доставка', 'Скидка', 'Акция']].map(lambda x: 1 if x > 0 else 0)
# Q-критерий Кохрена
from statsmodels.stats import contingency_tables
print(contingency_tables.cochrans_q(All[['Доставка', 'Скидка', 'Акция']]))
# p_value=0.56 > 0.05 - гипотезу о том, что все 3 акции дали одинаковый эффект не отвергаем

df          2
pvalue      0.5580351457700471
statistic   1.1666666666666667


In [191]:
# Определим индексы разнообразия заказов по категории товара (без удаления дубликатов номеров заказа),
# регионам (удалим дубликаты номеров заказа), по типам доставки (удалим дубликаты номеров заказа)
import numpy as np
Orders=pd.read_excel("Sample - Superstore.xlsx",sheet_name='Orders')
Category = Orders['Category'].value_counts().reset_index()
Region = Orders[['Order ID', 'Region']].drop_duplicates(subset=['Order ID'])['Region'].value_counts().reset_index()
Ship_Mode = Orders[['Order ID', 'Ship Mode']].drop_duplicates(subset=['Order ID'])['Ship Mode'].value_counts().reset_index()
Category.columns = ['Category', 'Count']
Region.columns = ['Region', 'Count']
Ship_Mode.columns = ['Ship Mode', 'Count']
Category['Относ.частота'] = Category['Count']/Category['Count'].sum()
Region['Относ.частота'] = Region['Count']/Region['Count'].sum()
Ship_Mode['Относ.частота'] = Ship_Mode['Count']/Ship_Mode['Count'].sum()
# Максимальная энтропия
print(f"Максимальная энтропия Category: {np.log(len(Category))}")
print(f"Максимальная энтропия Region: {np.log(len(Region))}")
print(f"Максимальная энтропия Ship_Mode: {np.log(len(Ship_Mode))}")
# Индексы разнообразия
print(f"Category: {scipy.stats.entropy(Category['Относ.частота'], base=None)}")
print(f"Region: {scipy.stats.entropy(Region['Относ.частота'], base=None)}")
print(f"Ship_Mode: {scipy.stats.entropy(Ship_Mode['Относ.частота'], base=None)}")
# Оценка равномерности (относительная энтропия)
print(f"Category равномерность: {scipy.stats.entropy(Category['Относ.частота']) / np.log(len(Category)):.2%}")
print(f"Region равномерность: {scipy.stats.entropy(Region['Относ.частота']) / np.log(len(Region)):.2%}")
print(f"Ship_Mode равномерность: {scipy.stats.entropy(Ship_Mode['Относ.частота']) / np.log(len(Ship_Mode)):.2%}")
# Регионы распределены почти идеально равномерно, категории - умеренно равномерно, типы доставки - наименее равномерно.

Максимальная энтропия Category: 1.0986122886681098
Максимальная энтропия Region: 1.3862943611198906
Максимальная энтропия Ship_Mode: 1.3862943611198906
Category: 0.9459905988188129
Region: 1.3578968309575594
Ship_Mode: 1.0706488494030282
Category равномерность: 86.11%
Region равномерность: 97.95%
Ship_Mode равномерность: 77.23%


In [212]:
# По выборке за 2017 год оценим доли заказов по типу доставки во всей генеральной совокупности
# Построим доверительные интервалы и проверим, попадают ли реальные доли ГС в эти доверительные интервалы.
import pandas as pd
from statsmodels.stats.proportion import proportion_confint
Orders0=pd.read_excel("Sample - Superstore.xlsx",sheet_name="Orders")
Orders=Orders0[pd.to_datetime(Orders0['Order Date']).dt.year==2017].drop_duplicates(subset=['Order ID']).groupby('Ship Mode')['Order ID'].count().reset_index(name='Count')
Orders['Доля']=Orders['Count']/Orders['Count'].sum()
print("Доли: ", Orders)
for i, row in Orders.iterrows():
    print(f"Доверительный интервал: {row['Ship Mode']}, {proportion_confint(count=row['Count'], nobs=Orders['Count'].sum(), alpha=0.05)}" )
# Доверительный интервал: First Class, (0.1504910278927442, 0.18620132539711948) - 0.16 попадает
# Доверительный интервал: Same Day, (0.042088965439172205, 0.06342377907772169) - 0.052 попадает
# Доверительный интервал: Second Class, (0.17325958724867851, 0.21085422425102512) - 0.19 попадает
# Доверительный интервал: Standard Class, (0.5633437056775266, 0.6103373850160123) - 0.58 попадает
# Выборка за 2017 год репрезентативна

Доли:          Ship Mode  Count      Доля
0     First Class    284  0.168346
1        Same Day     89  0.052756
2    Second Class    324  0.192057
3  Standard Class    990  0.586841
Доверительный интервал: First Class, (0.1504910278927442, 0.18620132539711948)
Доверительный интервал: Same Day, (0.042088965439172205, 0.06342377907772169)
Доверительный интервал: Second Class, (0.17325958724867851, 0.21085422425102512)
Доверительный интервал: Standard Class, (0.5633437056775266, 0.6103373850160123)
